In [1]:
import torch
import torch.nn as nn
from torch.func import jacrev, jacfwd, vmap

import sys
sys.path.append("..")

from src.model import *

In [2]:
# Load the trained real-valued model
model = PolyNetwork(
    input_dim=2,
    output_dim=2,
    hidden_dims=[8, 6],
    polynomial_degree=3,
)

model.load_state_dict(torch.load("./saved_models/polynomial_nn_model.pth"))

# Convert to complex-differentiable CR version
cr_model = CRPolyNetwork.from_polynetwork(model)

In [3]:
# Test: Verify that CR model matches original model on real inputs
model = model.double()
cr_model = cr_model.double()
model.eval()
cr_model.eval()

# Real input for original model
x_real = torch.tensor([[0.5, -1.2], [1.0, 0.5], [-0.3, 0.8]])
x_real = x_real.double()

# Get original model output
with torch.no_grad():
    output_real = model(x_real)

# Convert to stacked [real; imag] format for CR model
# For real inputs, imaginary part is zero
x_imag = torch.zeros_like(x_real)
x_imag = x_imag.double()
z = c_join(x_real, x_imag)  # Shape: (3, 4) = (batch, 2*input_dim)

# Get CR model output
with torch.no_grad():
    output_cr = cr_model(z)

# Extract real and imaginary parts from CR output
output_cr_real, output_cr_imag = c_split(output_cr)

# Verify outputs match
assert torch.allclose(output_real, output_cr_real, atol=1e-6), "CR model output does not match original model output on real inputs."
assert torch.allclose(output_cr_imag, torch.zeros_like(output_cr_imag), atol=1e-6), "CR model imaginary output is not zero on real inputs."

In [4]:
# Test: Compute Jacobian of CR model at real inputs
jac_cr = vmap(jacfwd(cr_model))
j_cr = jac_cr(z)
print("\nJacobian shape:", j_cr.shape)
print(j_cr)


Jacobian shape: torch.Size([3, 4, 4])
tensor([[[ 2.5431e+00, -1.8720e+00,  5.9011e-16, -5.6701e-16],
         [-6.5169e+00,  6.2716e+00, -2.9647e-15,  2.5205e-15],
         [-5.9011e-16,  5.6701e-16,  2.5431e+00, -1.8720e+00],
         [ 2.9647e-15, -2.5205e-15, -6.5169e+00,  6.2716e+00]],

        [[ 2.7436e+00, -3.7751e+00,  2.0174e-16, -5.8204e-16],
         [-5.5209e+00,  6.0165e+00, -1.7193e-15,  1.5269e-15],
         [-2.0174e-16,  5.8204e-16,  2.7436e+00, -3.7751e+00],
         [ 1.7193e-15, -1.5269e-15, -5.5209e+00,  6.0165e+00]],

        [[ 5.6364e+00, -5.2213e+00,  3.7050e-16, -6.4449e-16],
         [-6.2207e+00,  6.4860e+00, -1.0054e-15,  1.1262e-15],
         [-3.7050e-16,  6.4449e-16,  5.6364e+00, -5.2213e+00],
         [ 1.0054e-15, -1.1262e-15, -6.2207e+00,  6.4860e+00]]],
       dtype=torch.float64, grad_fn=<ViewBackward0>)


In [5]:
j_real_cr = (j_cr[:,:2,:2] + j_cr[:, 2:, 2:])/2
j_imag_cr = (j_cr[:,2:,:2] - j_cr[:, :2, 2:])/2
print("Jacobian of real part : ", j_real_cr)
print("Jacobian of imag part : ", j_imag_cr)

Jacobian of real part :  tensor([[[ 2.5431, -1.8720],
         [-6.5169,  6.2716]],

        [[ 2.7436, -3.7751],
         [-5.5209,  6.0165]],

        [[ 5.6364, -5.2213],
         [-6.2207,  6.4860]]], dtype=torch.float64, grad_fn=<DivBackward0>)
Jacobian of imag part :  tensor([[[-5.9011e-16,  5.6701e-16],
         [ 2.9647e-15, -2.5205e-15]],

        [[-2.0174e-16,  5.8204e-16],
         [ 1.7193e-15, -1.5269e-15]],

        [[-3.7050e-16,  6.4449e-16],
         [ 1.0054e-15, -1.1262e-15]]], dtype=torch.float64,
       grad_fn=<DivBackward0>)


In [6]:
# Compute Jacobian of original model at real inputs
jac = vmap(jacfwd(model))
j = jac(x_real)
print("\nJacobian shape:", j.shape)
print(j)


Jacobian shape: torch.Size([3, 2, 2])
tensor([[[ 2.5431, -1.8720],
         [-6.5169,  6.2716]],

        [[ 2.7436, -3.7751],
         [-5.5209,  6.0165]],

        [[ 5.6364, -5.2213],
         [-6.2207,  6.4860]]], dtype=torch.float64, grad_fn=<ViewBackward0>)


In [7]:
assert torch.allclose(j_real_cr, j, atol=1e-10), "Jacobian of real part does not match original model Jacobian"